# KVO_04 — Final multisite UAV U-Net and probability maps

Production model using all four sites and crowns ≥5 m². Stage A uses spatial 100 m validation blocks only to select epoch count. Stage B refits a fresh U-Net for that fixed epoch count on all eligible crown labels. It then generates continuous wall-to-wall Musanga probability rasters for all four orthomosaics. KVO_03 remains the independent LOSO validation.


In [ ]:
!pip -q install geopandas rasterio pyogrio segmentation-models-pytorch scikit-learn matplotlib tqdm
from pathlib import Path
import json, hashlib, random, shutil, time, warnings, gc
import numpy as np, pandas as pd, geopandas as gpd, matplotlib.pyplot as plt
import rasterio
from rasterio.features import rasterize
from rasterio.windows import Window
from sklearn.metrics import roc_auc_score, average_precision_score
from tqdm.auto import tqdm
import torch, torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import segmentation_models_pytorch as smp
from google.colab import drive
warnings.filterwarnings('ignore')
SEED=42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
DEVICE='cuda' if torch.cuda.is_available() else 'cpu'; torch.backends.cudnn.benchmark=True
print('DEVICE:',DEVICE); print(torch.cuda.get_device_name(0) if DEVICE=='cuda' else '')

In [ ]:
DRIVE_ROOT=Path('/content/drive'); MYDRIVE=DRIVE_ROOT/'MyDrive'
def healthy():
    try: return MYDRIVE.exists() and any(MYDRIVE.iterdir())
    except OSError: return False
if not healthy(): drive.mount(str(DRIVE_ROOT),force_remount=True)
ROOT=MYDRIVE/'Colab Notebooks'/'Musanga'; GEO=ROOT/'data'/'Geo_Ref'; AD=GEO/'Annotations'
OUT=ROOT/'KVO_work'/'KVO_04_final_multisite'; CACHE=OUT/'cache'; MAPS=OUT/'probability_maps'
for p in [OUT,CACHE,MAPS]: p.mkdir(parents=True,exist_ok=True)
SITES={
 'MOS_1_2':{'raster':GEO/'MOS_1_2_kvo.tif','annotations':AD/'Tree_Annotations_MOS_1_2_kvo.shp'},
 'Yoko':{'raster':GEO/'Yoko_kvo_georef.tif','annotations':AD/'Tree_Annotations_Yoko_kvo_georef.shp'},
 'Baego':{'raster':GEO/'Baego_georef.tif','annotations':AD/'Tree_Annotation_Baego.shp'},
 'Yangambi':{'raster':GEO/'Yangambi_georef.tif','annotations':AD/'Tree_Annotation_Yangambi_georef.shp'}}
for s,d in SITES.items(): assert d['raster'].exists() and d['annotations'].exists(),s
PATCH=256; STRIDE=256; IGNORE=255; BLOCK_M=100.; VAL_FRAC=.20; MIN_CROWN_AREA_M2=5.0
MEAN=np.array([0.485,0.456,0.406],np.float32)[:,None,None]; STD=np.array([0.229,0.224,0.225],np.float32)[:,None,None]
FORCE_CACHE=False; FORCE_TUNE=False; FORCE_REFIT=False; FORCE_MAPS=False
print('OUT:',OUT)


## 1 — Audit filtered crowns


In [ ]:
ANNS={}; audit=[]
for site,d in SITES.items():
    with rasterio.open(d['raster']) as src: crs=src.crs; bounds=src.bounds; gsd=(abs(src.transform.a),abs(src.transform.e)); dtype=src.dtypes[0]
    assert dtype=='uint8',f'{site}: expected uint8 RGB, got {dtype}'
    g=gpd.read_file(d['annotations']); assert 'Class_ID' in g.columns
    if g.crs!=crs: g=g.to_crs(crs)
    g=g[g.geometry.notna() & ~g.geometry.is_empty].copy(); g['class_id_num']=pd.to_numeric(g.Class_ID,errors='coerce'); g=g[g.class_id_num.notna()].copy(); g['class_id_num']=g.class_id_num.astype(int)
    g['target']=(g.class_id_num==1).astype(np.uint8)
    if g.crs is None or not g.crs.is_projected: raise ValueError(f'{site}: projected CRS required for m² crown filtering; got {g.crs}')
    g['crown_area_m2']=g.geometry.area.astype(float)
    n_before=len(g); mus_before=int(g.target.sum()); other_before=int((1-g.target).sum())
    g=g[g.crown_area_m2>=MIN_CROWN_AREA_M2].copy().reset_index(drop=True)
    g['crown_id']=np.arange(len(g),dtype=np.int64); c=g.geometry.centroid
    g['block_id']=np.floor((c.x-bounds.left)/BLOCK_M).astype(int).astype(str)+'_'+np.floor((c.y-bounds.bottom)/BLOCK_M).astype(int).astype(str)
    ANNS[site]=g; audit.append({'site':site,'min_crown_area_m2':MIN_CROWN_AREA_M2,'n_before':n_before,'removed_lt_5m2':n_before-len(g),'n_crowns':len(g),'musanga':int(g.target.sum()),'other':int((1-g.target).sum()),'musanga_removed':mus_before-int(g.target.sum()),'other_removed':other_before-int((1-g.target).sum()),'blocks':g.block_id.nunique(),'gsd_x':gsd[0],'gsd_y':gsd[1]})
audit_df=pd.DataFrame(audit); display(audit_df); audit_df.to_csv(OUT/'site_audit.csv',index=False)

def train_val_assignment(g,seed):
    blocks=g.block_id.unique().copy(); rng=np.random.default_rng(seed)
    for attempt in range(1000):
        b=blocks.copy(); rng.shuffle(b); nv=max(1,int(round(VAL_FRAC*len(b)))); vb=set(b[:nv]); sp=np.where(g.block_id.isin(vb),'val','train')
        if all(g.loc[sp==q,'target'].nunique()==2 for q in ['train','val']): return pd.Series(sp,index=g.index)
    raise RuntimeError('Could not make two-class train/val split')

## 2 — Reusable all-site caches


In [ ]:
def train_val_assignment(g,seed):
    blocks=g.block_id.unique().copy(); rng=np.random.default_rng(seed)
    for _ in range(1000):
        b=blocks.copy(); rng.shuffle(b); nv=max(1,int(round(VAL_FRAC*len(b)))); vb=set(b[:nv])
        sp=np.where(g.block_id.isin(vb),'val','train')
        if all(g.loc[sp==q,'target'].nunique()==2 for q in ['train','val']): return pd.Series(sp,index=g.index)
    raise RuntimeError('Could not make two-class split')

def build_cache(mode='tune',force=False):
    assert mode in ['tune','all']; split_ann={}
    for i,s in enumerate(SITES):
        g=ANNS[s].copy(); g['split']=train_val_assignment(g,SEED+100*i) if mode=='tune' else 'train'; split_ann[s]=g
    payload={'mode':mode,'patch':PATCH,'block_m':BLOCK_M,'val_frac':VAL_FRAC,'min_area':MIN_CROWN_AREA_M2,
      'files':{s:(SITES[s]['raster'].stat().st_size,SITES[s]['annotations'].stat().st_size,int(SITES[s]['annotations'].stat().st_mtime)) for s in SITES}}
    fp=hashlib.sha1(json.dumps(payload,sort_keys=True).encode()).hexdigest()[:12]; path=CACHE/f'{mode}_patches_{fp}.npz'
    if path.exists() and not force:
        print('Reusing cache:',path); d=np.load(path); return d['X'],d['Y'],d['S'],path
    X=[]; Y=[]; SS=[]; tmp=Path('/content/musanga_kvo04'); tmp.mkdir(exist_ok=True)
    for site in SITES:
        g=split_ann[site]; rp=SITES[site]['raster']
        with rasterio.open(rp) as src:
            H,W=src.height,src.width; splits=['train','val'] if mode=='tune' else ['train']; mps={}
            for sp in splits:
                mp=tmp/f'{mode}_{site}_{sp}.tif'; prof=src.profile.copy(); prof.update(count=1,dtype='uint8',nodata=IGNORE,compress='lzw',tiled=True)
                a=g[g.split==sp]
                m=rasterize([(z,int(y)) for z,y in zip(a.geometry,a.target)],out_shape=(H,W),transform=src.transform,fill=IGNORE,dtype='uint8')
                with rasterio.open(mp,'w',**prof) as dst: dst.write(m,1)
                del m; gc.collect(); mps[sp]=mp
            rr={sp:rasterio.open(mp) for sp,mp in mps.items()}
            try:
                for r0 in tqdm(range(0,H,PATCH),desc=f'{mode}: cache {site}'):
                    for c0 in range(0,W,PATCH):
                        hh=min(PATCH,H-r0); ww=min(PATCH,W-c0); w=Window(c0,r0,ww,hh)
                        mm={sp:rr[sp].read(1,window=w) for sp in splits}; hit={sp:np.any(mm[sp]!=IGNORE) for sp in splits}
                        if mode=='tune':
                            if hit['train']==hit['val']: continue
                            sp='train' if hit['train'] else 'val'; m=mm[sp]; sc=0 if sp=='train' else 1
                        else:
                            if not hit['train']: continue
                            m=mm['train']; sc=0
                        arr=src.read([1,2,3],window=w,boundless=True,fill_value=0,out_shape=(3,PATCH,PATCH))
                        y=np.full((PATCH,PATCH),IGNORE,np.uint8); y[:hh,:ww]=m
                        X.append(arr.astype(np.uint8)); Y.append(y); SS.append(sc)
            finally:
                for q in rr.values(): q.close()
                for mp in mps.values(): mp.unlink(missing_ok=True)
    X=np.stack(X); Y=np.stack(Y); SS=np.asarray(SS,np.uint8); np.savez_compressed(path,X=X,Y=Y,S=SS)
    print('Saved:',path,'train/val:',int((SS==0).sum()),int((SS==1).sum()))
    return X,Y,SS,path


## 3 — Select epoch count, then refit on all four sites


In [ ]:
class DS(Dataset):
    def __init__(self,X,Y,S,code=0,aug=False): self.X=X; self.Y=Y; self.idx=np.where(S==code)[0]; self.aug=aug
    def __len__(self): return len(self.idx)
    def __getitem__(self,j):
        i=self.idx[j]; x=self.X[i].astype(np.float32)/255.; y=self.Y[i].copy()
        if self.aug:
            if np.random.rand()<.5: x=x[:,:,::-1].copy(); y=y[:,::-1].copy()
            if np.random.rand()<.5: x=x[:,::-1,:].copy(); y=y[::-1,:].copy()
            k=np.random.randint(4); x=np.rot90(x,k,axes=(1,2)).copy(); y=np.rot90(y,k).copy(); x=np.clip(x*np.random.uniform(.9,1.1),0,1)
        return torch.from_numpy((x-MEAN)/STD),torch.from_numpy(y.astype(np.int64))
def model_new(): return smp.Unet(encoder_name='resnet18',encoder_weights='imagenet',in_channels=3,classes=1).to(DEVICE)
def score(model,dl):
    yy=[]; pp=[]; model.eval()
    with torch.inference_mode():
        for x,y in dl:
            x=x.to(DEVICE); y=y.to(DEVICE)
            with torch.amp.autocast('cuda',enabled=DEVICE=='cuda'): p=torch.sigmoid(model(x).squeeze(1))
            v=y!=IGNORE; yy.append(y[v].cpu().numpy()); pp.append(p[v].float().cpu().numpy())
    y=np.concatenate(yy); p=np.concatenate(pp); return roc_auc_score(y,p),average_precision_score(y,p)
def fit(model,dl,n,va=None,ck=None,hp=None,early=False):
    opt=torch.optim.AdamW(model.parameters(),lr=2e-4,weight_decay=1e-4); scaler=torch.amp.GradScaler('cuda',enabled=DEVICE=='cuda')
    best=-1; be=None; wait=0; hist=[]
    for ep in range(1,n+1):
        model.train(); ls=[]
        for x,y in dl:
            x=x.to(DEVICE); y=y.to(DEVICE); opt.zero_grad(set_to_none=True)
            with torch.amp.autocast('cuda',enabled=DEVICE=='cuda'):
                z=model(x).squeeze(1); v=y!=IGNORE; loss=nn.functional.binary_cross_entropy_with_logits(z[v],y[v].float())
            scaler.scale(loss).backward(); scaler.step(opt); scaler.update(); ls.append(loss.item())
        row={'epoch':ep,'loss':float(np.mean(ls))}
        if va is not None:
            auc,ap=score(model,va); row.update(val_auc=auc,val_ap=ap); print(f'tune ep {ep:02d} loss={row["loss"]:.4f} AUC={auc:.4f} AP={ap:.4f}')
            if ap>best: best=ap; be=ep; wait=0; torch.save(model.state_dict(),ck)
            else: wait+=1
        else: print(f'refit ep {ep:02d}/{n} loss={row["loss"]:.4f}')
        hist.append(row)
        if hp: pd.DataFrame(hist).to_csv(hp,index=False)
        if early and wait>=6: break
    return be

Xt,Yt,St,tc=build_cache('tune',FORCE_CACHE); tid=tc.stem.replace('tune_patches_','')
bj=OUT/f'best_epoch_{tid}.json'; tck=OUT/f'tune_best_{tid}.pt'; th=OUT/f'tune_history_{tid}.csv'; bs=16 if DEVICE=='cuda' else 4
tr=DataLoader(DS(Xt,Yt,St,0,True),batch_size=bs,shuffle=True,num_workers=2,pin_memory=DEVICE=='cuda',persistent_workers=True,prefetch_factor=2)
va=DataLoader(DS(Xt,Yt,St,1,False),batch_size=bs,shuffle=False,num_workers=2,pin_memory=DEVICE=='cuda',persistent_workers=True,prefetch_factor=2)
if bj.exists() and tck.exists() and not FORCE_TUNE:
    BEST_EPOCH=int(json.loads(bj.read_text())['best_epoch']); print('Reusing selected epoch:',BEST_EPOCH)
else:
    m=model_new(); BEST_EPOCH=int(fit(m,tr,30,va,tck,th,True)); bj.write_text(json.dumps({'best_epoch':BEST_EPOCH,'selection':'max spatial-validation AP'},indent=2)); del m
del Xt,Yt,St,tr,va; gc.collect(); torch.cuda.empty_cache()
print('Selected refit epochs:',BEST_EPOCH)

Xa,Ya,Sa,ac=build_cache('all',FORCE_CACHE); aid=ac.stem.replace('all_patches_','')+f'_ep{BEST_EPOCH:02d}'
FINAL_CK=OUT/f'final_unet_resnet18_{aid}.pt'; FH=OUT/f'final_refit_history_{aid}.csv'
if FINAL_CK.exists() and FH.exists() and not FORCE_REFIT:
    print('Existing final model — SKIPPING REFIT'); final_model=model_new(); final_model.load_state_dict(torch.load(FINAL_CK,map_location=DEVICE))
else:
    dl=DataLoader(DS(Xa,Ya,Sa,0,True),batch_size=bs,shuffle=True,num_workers=2,pin_memory=DEVICE=='cuda',persistent_workers=True,prefetch_factor=2)
    final_model=model_new(); fit(final_model,dl,BEST_EPOCH,None,None,FH,False); torch.save(final_model.state_dict(),FINAL_CK); print('Saved:',FINAL_CK)
del Xa,Ya,Sa; gc.collect(); torch.cuda.empty_cache()


## 4 — Wall-to-wall continuous probability maps

Output encoding: uint16 0–10000 = probability 0–1; 65535 = NoData. Core/halo inference avoids tile-edge artifacts. Source dataset mask/alpha is respected, so transparent and NoData areas are not predicted. Completed site rasters are persisted to Drive and reused.


In [ ]:
CORE=512; HALO=64; SCALE=10000; OUT_NODATA=65535
def output_ok(p,sp):
    if not p.exists(): return False
    try:
        with rasterio.open(sp) as a,rasterio.open(p) as b: return b.width==a.width and b.height==a.height and b.crs==a.crs and b.transform==a.transform and b.dtypes[0]=='uint16'
    except: return False
def infer_site(model,site,force=False):
    sp=SITES[site]['raster']; op=MAPS/f'{site}_musanga_probability_uint16.tif'
    if output_ok(op,sp) and not force: print('Existing map — SKIPPING:',op); return op
    lp=Path('/content')/op.name; lp.unlink(missing_ok=True); model.eval()
    with rasterio.open(sp) as src:
        prof=src.profile.copy(); prof.update(count=1,dtype='uint16',nodata=OUT_NODATA,compress='deflate',predictor=2,tiled=True,bigtiff='YES')
        with rasterio.open(lp,'w',**prof) as dst:
            for r0 in tqdm(range(0,src.height,CORE),desc=f'MAP {site}'):
                for c0 in range(0,src.width,CORE):
                    hh=min(CORE,src.height-r0); ww=min(CORE,src.width-c0); win=Window(c0-HALO,r0-HALO,ww+2*HALO,hh+2*HALO)
                    a=src.read([1,2,3],window=win,boundless=True,fill_value=0).astype(np.float32)/255.
                    ph=(32-a.shape[1]%32)%32; pw=(32-a.shape[2]%32)%32
                    if ph or pw: a=np.pad(a,((0,0),(0,ph),(0,pw)))
                    x=torch.from_numpy((a-MEAN)/STD).unsqueeze(0).to(DEVICE)
                    with torch.inference_mode(),torch.amp.autocast('cuda',enabled=DEVICE=='cuda'): p=torch.sigmoid(model(x).squeeze()).float().cpu().numpy()
                    p=p[:int(win.height),:int(win.width)][HALO:HALO+hh,HALO:HALO+ww]
                    cw=Window(c0,r0,ww,hh); valid=src.dataset_mask(window=cw)>0
                    o=np.full((hh,ww),OUT_NODATA,np.uint16); o[valid]=np.rint(np.clip(p[valid],0,1)*SCALE).astype(np.uint16); dst.write(o,1,window=cw)
            dst.update_tags(MUSANGA_SCALE=SCALE,MODEL=FINAL_CK.name,MIN_CROWN_AREA_M2=MIN_CROWN_AREA_M2,TRAINING_SITES=','.join(SITES))
    shutil.copy2(lp,op); lp.unlink(missing_ok=True); print('Saved:',op); return op
maps={s:infer_site(final_model,s,FORCE_MAPS) for s in SITES}


## 5 — Manifest and sampled QA


In [ ]:
rows=[]
for s,p in maps.items():
    with rasterio.open(p) as q:
        a=q.read(1,out_shape=(max(1,q.height//20),max(1,q.width//20))); v=a!=OUT_NODATA
        rows.append({'site':s,'path':str(p),'width':q.width,'height':q.height,'valid_sample_pixels':int(v.sum()),
                     'mean_probability_sample':float((a[v]/SCALE).mean()) if v.any() else np.nan,
                     'p95_probability_sample':float(np.quantile(a[v]/SCALE,.95)) if v.any() else np.nan})
qa=pd.DataFrame(rows); display(qa); qa.to_csv(OUT/'probability_map_QA.csv',index=False)
manifest={'model':str(FINAL_CK),'best_epoch':BEST_EPOCH,'minimum_crown_area_m2':MIN_CROWN_AREA_M2,'training_sites':list(SITES),
          'encoding':'uint16 0..10000 = probability 0..1; 65535 = NoData','maps':{s:str(p) for s,p in maps.items()},
          'validation_note':'Independent site-generalization evidence is KVO_03 LOSO; KVO_04 is final all-data production fit.'}
(OUT/'production_manifest.json').write_text(json.dumps(manifest,indent=2)); print(json.dumps(manifest,indent=2)); print('DONE')
